<a href="https://colab.research.google.com/github/vermasachin6102/JoyAI-Echo/blob/main/joyai_echo_profiling_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JoyAI-Echo — PROFILING LAB (Instrumented Inference)
**Purpose:** Maximum observability for optimizing the video+audio generation pipeline. Clone of `seed_veo_3__joy_ai_echo_v1.ipynb` but every stage, swap, kernel, and GPU counter is logged.

**What you get vs the seed notebook:**
| Seed notebook | Profiling lab |
|---|---|
| wall time per run | wall time **per stage, per shot, per denoise step, per stage-swap, per decode tile** |
| 1 GPU line | continuous GPU telemetry (mem, util, temp, power, clocks @ 1 Hz) + `torch.cuda` stats |
| "FP8 ON/OFF" | dtype element histogram + per-layer dtype after `fp8_cast` |
| offload ON/OFF | `n_blocks`, resident GB, **H2D ms per block per step** |
| tiling ON/OFF | effective #tiles, tile geometry, **ms per tile**, overlap cost |
| latent shapes hidden | `tokens = ((F-1)//8+1)*(H//32)*(W//32)` + `video_shape` + `audio_shape` printed |
| no JSON artifact | `profiling_report_<timestamp>.json` + `gpu_timeline.csv` + `per_step.csv` |

> **Cost:** profiling adds ~2-5% overhead (profiler + NVML polling). Disable `ENABLE_TORCH_PROFILER` for a clean wall-time run.

**Workflow:** Run cells top → bottom. Cell 7 does one real generation and writes all artifacts to `inference_result/profiling/`. Cell 8 visualizes. Cell 9 (optional) runs a single-step `torch.profiler` kernel breakdown.


In [ ]:
!nvidia-smi
print("\n" + "="*70)
import subprocess, json, sys
def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.stdout.strip()
r = subprocess.run(["nvidia-smi","--query-gpu=name,driver_version,memory.total,memory.free,memory.used,utilization.gpu,utilization.memory,temperature.gpu,power.draw,clocks.sm,clocks.mem","--format=csv,noheader,nounits"], capture_output=True, text=True, check=True)
cols = [x.strip() for x in r.stdout.strip().split(",")]
labels = ["GPU","Driver","VRAM total MiB","free MiB","used MiB","GPU util %","Mem util %","Temp C","Power W","SM clock MHz","Mem clock MHz"]
for k,v in zip(labels, cols):
    print(f"{k:18s}: {v}")
try:
    vram_gb = float(cols[2])/1024
except: vram_gb = 22.0
_name = cols[0]
print(f"\nParsed: {_name} | {vram_gb:.1f} GB VRAM")
print("\n--- nvidia-smi -q (clipped) ---")
print(sh(["nvidia-smi","-q"])[:2500])
if vram_gb < 22:
    print("\nWARNING: <22GB VRAM -- too small for either stage. Switch to L4/A100.")
elif vram_gb < 40:
    print("\n24GB-class (L4): quantized paths recommended.")
    print("   Stage1: Q4_0 GGUF -> NF4 (~13GB peak)")
    print("   Stage2: FP8 downcast (~18GB vs 36.5GB bf16)")
else:
    print(f"\n{vram_gb:.0f}GB-class: can run full bf16 if desired, but profiling lab defaults to fp8+offload=auto.")
print(f"\n[vram_gb={vram_gb:.1f}] exported for later cells")


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    print("Drive mounted")
except Exception as e:
    print("Drive mount skipped/failed:", e)
import os, sys
from pathlib import Path
REPO_ROOT = "/content/JoyAI-Echo"
OUTPUT_DIR = f"{REPO_ROOT}/inference_result"
LOCAL_HF_CACHE = "/content/hf_cache_local"
PROFILING_DIR = Path(f"{REPO_ROOT}/inference_result/profiling")
for p in [LOCAL_HF_CACHE, OUTPUT_DIR, PROFILING_DIR]:
    os.makedirs(p, exist_ok=True)
    print(f"  {p}")
# Profiling knobs — tweak here
ENABLE_TORCH_PROFILER = False
ENABLE_GPU_POLLING   = True
GPU_POLL_INTERVAL    = 1.0
SAVE_TORCH_PROFILER_TRACE = True
print(f"\nProfiling knobs: profiler={ENABLE_TORCH_PROFILER} polling={ENABLE_GPU_POLLING} interval={GPU_POLL_INTERVAL}s")
import datetime
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID={RUN_ID}  (used for artifact names)")


In [ ]:
import os, sys, glob, subprocess, shutil, json, time
from pathlib import Path
os.chdir("/content")
REPO_ROOT = "/content/JoyAI-Echo"
OUTPUT_DIR = f"{REPO_ROOT}/inference_result"
LOCAL_HF_CACHE = "/content/hf_cache_local"
PROFILING_DIR = f"{REPO_ROOT}/inference_result/profiling"
os.makedirs(LOCAL_HF_CACHE, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(PROFILING_DIR, exist_ok=True)
def run(cmd, cwd=None, label=""):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"--- FAILED: {label or ' '.join(cmd)} (exit {r.returncode}) ---")
        print(r.stdout[-2000:]); print(r.stderr[-4000:])
        raise RuntimeError(f"Step failed: {label or cmd}")
    return r
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("hf")
    print("HF token from Colab secret 'hf'")
except Exception as e:
    print("HF secret 'hf' not found, gated repos will fail:", e)
if not os.path.exists(f"{REPO_ROOT}/requirements.txt"):
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    print("Cloning repo...")
    run(["git","clone","https://github.com/vermasachin6102/JoyAI-Echo.git", REPO_ROOT], label="git clone")
else:
    print("Repo present, skipping clone.")
for sub in ["ltx-core/src","ltx-pipelines/src","ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path: sys.path.insert(0, p)
print("Installing ffmpeg + profiling utils (psutil, pynvml)...")
run(["apt-get","-qq","update"], label="apt update")
run(["apt-get","-qq","install","-y","ffmpeg"], label="ffmpeg")
print("Installing pinned torch stack (cu128, 2.8.0) — takes ~3 min...")
run(["pip","install","--quiet","--force-reinstall","--index-url","https://download.pytorch.org/whl/cu128",
     "torch==2.8.0","torchvision==0.23.0","torchaudio==2.8.0"], label="torch install")
import torch as _chk
if not _chk.__version__.startswith("2.8.0"):
    raise RuntimeError(f"torch is {_chk.__version__}, expected 2.8.0 — restart session and rerun from top")
print(f"torch {_chk.__version__} confirmed"); del _chk
print("Installing requirements.txt...")
run(["pip","install","--quiet","-r","requirements.txt"], cwd=REPO_ROOT, label="requirements.txt")
print("Installing huggingface_hub constrained...")
run(["pip","install","--quiet","huggingface_hub[cli]>=0.34.0,<1.0"], label="hf hub")
print("Installing profiling extras...")
run(["pip","install","--quiet","psutil","pynvml","matplotlib","pandas"], label="profiling extras")
import torch
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | available {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)} | cap {torch.cuda.get_device_capability(0)}")
from huggingface_hub import snapshot_download, hf_hub_download
os.makedirs(f"{REPO_ROOT}/checkpoints", exist_ok=True)
print("\nFetching JoyAI-Echo checkpoint (~46GB) to local disk...")
echo_dir = snapshot_download(repo_id="jdopensource/JoyAI-Echo", cache_dir=LOCAL_HF_CACHE, allow_patterns=["*.safetensors","*.json","*.md"])
cands = glob.glob(os.path.join(echo_dir,"**","*.safetensors"), recursive=True)
assert cands, "No .safetensors in jdopensource/JoyAI-Echo"
dst_echo = f"{REPO_ROOT}/checkpoints/echo-longvideo-release.safetensors"
if os.path.islink(dst_echo) or os.path.exists(dst_echo): os.remove(dst_echo)
os.symlink(cands[0], dst_echo)
print("Echo ->", os.path.realpath(dst_echo))
dst_test = f"{REPO_ROOT}/checkpoints/test.safetensors"
if os.path.islink(dst_test) or os.path.exists(dst_test): os.remove(dst_test)
os.symlink(dst_echo, dst_test)
print("Compat symlink test.safetensors ->", os.path.realpath(dst_test))
print("\nFetching gemma-3-12b-it (~24GB, gated)...")
gemma_dir = snapshot_download(repo_id="google/gemma-3-12b-it", cache_dir=LOCAL_HF_CACHE)
dst_gemma = f"{REPO_ROOT}/checkpoints/gemma-3-12b"
if os.path.islink(dst_gemma): os.remove(dst_gemma)
elif os.path.isdir(dst_gemma): shutil.rmtree(dst_gemma)
os.symlink(gemma_dir, dst_gemma)
print("Gemma ->", os.path.realpath(dst_gemma))
print("\nFetching gemma Q4_0 GGUF (~8GB, gated)...")
GGUF_PATH = hf_hub_download(repo_id="google/gemma-3-12b-it-qat-q4_0-gguf", filename="gemma-3-12b-it-q4_0.gguf", cache_dir=LOCAL_HF_CACHE)
print("GGUF ->", GGUF_PATH)
print("\n--- Verification ---")
ok=True
for label,path in [("test.safetensors",dst_test),(f"gemma shard 5",f"{dst_gemma}/model-00005-of-00005.safetensors"),("tokenizer",f"{dst_gemma}/tokenizer.model"),("GGUF",GGUF_PATH)]:
    exists=os.path.exists(path); print(("OK " if exists else "MISSING ")+label+f" — {path}"); ok=ok and exists
print("\nSETUP COMPLETE — go to next cell" if ok else "\nSetup incomplete — re-run this cell (downloads resume)")
Path("/content/gguf_path.txt").write_text(GGUF_PATH)
print(f"\nGGUF_PATH written to /content/gguf_path.txt")


In [ ]:
print("Pulling latest code...")
r = run(["git","pull","origin","main"], cwd=REPO_ROOT, label="git pull")
print(r.stdout.strip() or r.stderr.strip() or "(no output)")
import importlib.util
for _mod in ("gguf","bitsandbytes","psutil","pynvml"):
    if importlib.util.find_spec(_mod) is None:
        print(f"Installing missing dep: {_mod}")
        run(["pip","install","--quiet",_mod], label=f"install {_mod}")
for sub in ["ltx-core/src","ltx-pipelines/src","ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path: sys.path.insert(0, p)
print("Fast pull done")


In [ ]:
import time, json, csv, threading, subprocess, traceback, gc, os, sys
from pathlib import Path
from collections import Counter, defaultdict
import torch
try:
    import psutil
    HAS_PSUTIL = True
except: HAS_PSUTIL=False; print("psutil not available")
try:
    import pynvml
    pynvml.nvmlInit()
    HAS_PYNVML = True
    _nv_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    print(f"pynvml: {pynvml.nvmlSystemGetDriverVersion()} | {pynvml.nvmlDeviceGetName(_nv_handle)}")
except Exception as e:
    HAS_PYNVML=False; print(f"pynvml not available ({e}) — falling back to nvidia-smi parsing")
class ProfTimer:
    def __init__(self, label):
        self.label=label; self.t0=None; self.elapsed=None
    def __enter__(self):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        self.t0=time.perf_counter(); return self
    def __exit__(self, *a):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        self.elapsed=time.perf_counter()-self.t0
def gpu_mem_str():
    if not torch.cuda.is_available(): return {}
    return {"alloc_gb": round(torch.cuda.memory_allocated()/1024**3,3), "reserved_gb": round(torch.cuda.memory_reserved()/1024**3,3), "max_alloc_gb": round(torch.cuda.max_memory_allocated()/1024**3,3)}
def nvidia_smi_snapshot():
    if HAS_PYNVML:
        try:
            mem = pynvml.nvmlDeviceGetMemoryInfo(_nv_handle)
            util = pynvml.nvmlDeviceGetUtilizationRates(_nv_handle)
            temp = pynvml.nvmlDeviceGetTemperature(_nv_handle, pynvml.NVML_TEMPERATURE_GPU)
            power = pynvml.nvmlDeviceGetPowerUsage(_nv_handle)/1000.0
            clocks_sm = pynvml.nvmlDeviceGetClockInfo(_nv_handle, pynvml.NVML_CLOCK_SM)
            clocks_mem = pynvml.nvmlDeviceGetClockInfo(_nv_handle, pynvml.NVML_CLOCK_MEM)
            return {"mem_total_gb":round(mem.total/1024**3,2),"mem_used_gb":round(mem.used/1024**3,2),"mem_free_gb":round(mem.free/1024**3,2), "gpu_util":int(util.gpu),"mem_util":int(util.memory),"temp_c":int(temp),"power_w":round(power,1), "clock_sm_mhz":int(clocks_sm),"clock_mem_mhz":int(clocks_mem)}
        except Exception as e: return {"pynvml_err":str(e)}
    else:
        try:
            r=subprocess.run(["nvidia-smi","--query-gpu=memory.total,memory.used,memory.free,utilization.gpu,utilization.memory,temperature.gpu,power.draw,clocks.sm,clocks.mem","--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=3)
            cols=[x.strip() for x in r.stdout.strip().split(",")]
            return {"mem_total_gb":round(float(cols[0])/1024,2),"mem_used_gb":round(float(cols[1])/1024,2),"mem_free_gb":round(float(cols[2])/1024,2), "gpu_util":int(float(cols[3])),"mem_util":int(float(cols[4])),"temp_c":int(float(cols[5])),"power_w":float(cols[6]),"clock_sm_mhz":int(float(cols[7])),"clock_mem_mhz":int(float(cols[8]))}
        except Exception as e: return {"smi_err":str(e)}
class GPUMonitor(threading.Thread):
    def __init__(self, interval=1.0, log_path=None):
        super().__init__(daemon=True)
        self.interval=interval; self.log_path=Path(log_path) if log_path else None
        self.samples=[]; self._stop=threading.Event()
        self.t0=time.perf_counter()
    def run(self):
        if self.log_path:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_path,"w",newline="") as f:
                w=csv.writer(f); w.writerow(["elapsed_s","alloc_gb","reserved_gb","max_alloc_gb","mem_used_gb","gpu_util","mem_util","temp_c","power_w","clock_sm_mhz","clock_mem_mhz"])
        while not self._stop.is_set():
            elapsed=time.perf_counter()-self.t0
            mem=gpu_mem_str(); smi=nvidia_smi_snapshot()
            row={"elapsed_s":round(elapsed,2), **mem, **{k:v for k,v in smi.items() if k in ("mem_used_gb","gpu_util","mem_util","temp_c","power_w","clock_sm_mhz","clock_mem_mhz")}}
            self.samples.append(row)
            if self.log_path:
                with open(self.log_path,"a",newline="") as f:
                    w=csv.writer(f)
                    w.writerow([row.get(k,"") for k in ["elapsed_s","alloc_gb","reserved_gb","max_alloc_gb","mem_used_gb","gpu_util","mem_util","temp_c","power_w","clock_sm_mhz","clock_mem_mhz"]])
            time.sleep(self.interval)
    def stop(self):
        self._stop.set(); self.join(timeout=2)
def inspect_tiling(cfg, video_h, video_w, num_frames):
    info={"enabled":bool(cfg.tiled_decode_enabled) if hasattr(cfg,"tiled_decode_enabled") else False}
    if info["enabled"]:
        info.update({"tile_size_frames": int(cfg.tiled_decode_tile_size_frames), "tile_overlap_frames": int(cfg.tiled_decode_tile_overlap_frames), "tile_size_px": int(cfg.tiled_decode_tile_size_px), "tile_overlap_px": int(cfg.tiled_decode_tile_overlap_px)})
        eff_t = info["tile_size_frames"] - info["tile_overlap_frames"]
        eff_px = info["tile_size_px"] - info["tile_overlap_px"]
        n_t = max(1, -(-num_frames // max(1,eff_t)))
        n_h = max(1, -(-video_h // max(1,eff_px)))
        n_w = max(1, -(-video_w // max(1,eff_px)))
        info["estimated_tiles_t"]=n_t; info["estimated_tiles_h"]=n_h; info["estimated_tiles_w"]=n_w
        info["estimated_total_tiles"]=n_t*n_h*n_w
        info["overlap_cost_frames"]=round(info["tile_overlap_frames"]/max(1,info["tile_size_frames"]),3)
        info["overlap_cost_px"]=round(info["tile_overlap_px"]/max(1,info["tile_size_px"]),3)
    return info
def dtype_histogram(model):
    c=Counter()
    for p in model.parameters(): c[str(p.dtype)]+=p.numel()
    per_layer=[]
    for name,m in model.named_modules():
        if hasattr(m,"weight") and isinstance(m.weight, torch.nn.Parameter):
            per_layer.append((name, str(m.weight.dtype), m.weight.numel()))
    per_layer=sorted(per_layer, key=lambda x: -x[2])[:15]
    return dict(c), per_layer
print("Profiling infra loaded")
print(f"  HAS_PSUTIL={HAS_PSUTIL} HAS_PYNVML={HAS_PYNVML} torch={torch.__version__}")
print(f"  GPUMonitor, ProfTimer, gpu_mem_str, nvidia_smi_snapshot, inspect_tiling, dtype_histogram ready")
print(f"\n[test snapshot] gpu_mem={gpu_mem_str()} smi={nvidia_smi_snapshot()}")


In [ ]:
import json, glob, subprocess, time, threading, csv, traceback
from pathlib import Path
from collections import Counter
import torch
from IPython.display import Video, display
sys.path.insert(0, f"{REPO_ROOT}/ltx-core/src")
sys.path.insert(0, f"{REPO_ROOT}/ltx-pipelines/src")
sys.path.insert(0, f"{REPO_ROOT}/ltx-distillation/src")
PROFILING_DIR = Path(PROFILING_DIR)
PROFILING_DIR.mkdir(parents=True, exist_ok=True)
def generate_video_profiling(
    prompts,
    name="my_story",
    seed=42,
    num_frames=121,
    video_height=480,
    video_width=832,
    preview=True,
    seed_video=None,
    gemma_gguf=GGUF_PATH,
    fp8_generator=None,
    offload_blocks=None,
    enable_profiler=None,
    enable_gpu_polling=None,
):
    # Ensure PROFILING_DIR is Path even if Cell 2 was re-run as str
    PROFILING_DIR = Path(PROFILING_DIR)
    use_fp8 = (vram_gb < 40) if fp8_generator is None else bool(fp8_generator)
    do_profiler = ENABLE_TORCH_PROFILER if enable_profiler is None else bool(enable_profiler)
    do_polling = ENABLE_GPU_POLLING if enable_gpu_polling is None else bool(enable_gpu_polling)
    tokens = ((num_frames - 1)//8 + 1) * (video_height//32) * (video_width//32)
    adaln_gb = tokens * 9 * 4096 * 2 / 1024**3
    weights_gb = 18.1 if use_fp8 else 36.5
    headroom = vram_gb - weights_gb - 1.0
    use_offload = (adaln_gb > headroom) if offload_blocks is None else bool(offload_blocks)
    if use_offload: weights_gb=2.0; headroom=vram_gb - weights_gb - 1.0
    print("="*72)
    print(f"PROFILING RUN  name={name} seed={seed}  {num_frames} frames  {video_height}x{video_width} @25fps")
    print(f"   tokens={tokens:,}  (({num_frames}-1)//8+1)*({video_height}//32)*({video_width}//32)")
    print(f"   AdaLN needs ~{adaln_gb:.2f}GB | weights ~{weights_gb}GB | headroom ~{headroom:.1f}GB | offload={'ON' if use_offload else 'OFF'} fp8={'ON' if use_fp8 else 'OFF'}")
    print(f"   profiler={'ON' if do_profiler else 'OFF'}  gpu_polling={'ON' if do_polling else 'OFF'}")
    if adaln_gb > headroom:
        print(f"   WARNING LIKELY OOM — AdaLN {adaln_gb:.2f}GB > {headroom:.1f}GB headroom")
    print("="*72)
    prompts_dir = Path(REPO_ROOT)/"prompts"
    prompts_dir.mkdir(parents=True, exist_ok=True)
    prompt_file = prompts_dir / f"{name}.json"
    with open(prompt_file,"w") as f: json.dump({"prompts": prompts}, f, indent=2)
    print(f"Wrote {len(prompts)} shot(s) to {prompt_file}")
    run_id = time.strftime("%Y%m%d_%H%M%S")
    run_profiling_dir = Path(PROFILING_DIR) / f"{name}_{run_id}"
    run_profiling_dir.mkdir(parents=True, exist_ok=True)
    gpu_csv = run_profiling_dir / "gpu_timeline.csv"
    per_step_csv = run_profiling_dir / "per_step.csv"
    report_json = run_profiling_dir / "profiling_report.json"
    profiler_trace = run_profiling_dir / "profiler_trace.json"
    print(f"Profiling artifacts -> {run_profiling_dir}")
    monitor = None
    if do_polling:
        monitor = GPUMonitor(interval=GPU_POLL_INTERVAL, log_path=gpu_csv)
        monitor.t0 = time.perf_counter()
        monitor.start()
        print(f"GPU polling @ {GPU_POLL_INTERVAL}s -> {gpu_csv}")
    from inference import InferenceConfig, InferenceEngine
    import yaml
    cfg = InferenceConfig(
        f"{REPO_ROOT}/configs/inference.yaml",
        num_frames=num_frames, video_height=video_height, video_width=video_width, seed=seed,
        gemma_gguf_path=str(gemma_gguf) if gemma_gguf else None,
        quantization_fp8_enabled=use_fp8,
        sequential_offload_enabled=use_offload,
        prompts_glob=prompt_file.name,
        seed_video=str(seed_video) if seed_video else None,
    )
    tiling_info = inspect_tiling(cfg, video_height, video_width, num_frames)
    print(f"\n[Config] tiling={tiling_info}")
    (run_profiling_dir/"effective_config.json").write_text(json.dumps({"num_frames": cfg.num_frames, "video_height": cfg.video_height, "video_width": cfg.video_width, "video_fps": cfg.video_fps, "seed": cfg.seed, "v2a_grad_scale": cfg.v2a_grad_scale, "denoising_steps": list(cfg.denoising_steps), "denoising_sigmas": list(cfg.denoising_sigmas), "fp8": bool(cfg.quantization_fp8_enabled), "offload": bool(cfg.sequential_offload_enabled), "tiling": tiling_info, "tokens": tokens, "gemma_gguf": str(cfg.gemma_gguf_path), "seed_video": str(cfg.seed_video) if cfg.seed_video else None}, indent=2))
    sys_info = {"torch": torch.__version__, "cuda": torch.version.cuda, "python": sys.version.split()[0], "gpu_name": _name, "vram_gb": round(vram_gb,1), "smi_snapshot": nvidia_smi_snapshot()}
    if HAS_PSUTIL:
        vm=psutil.virtual_memory(); sys_info["ram_total_gb"]=round(vm.total/1024**3,1); sys_info["ram_available_gb"]=round(vm.available/1024**3,1)
        sys_info["cpu_count"]=psutil.cpu_count()
    print(f"[System] {json.dumps(sys_info, indent=2)}")
    overall_t0 = time.perf_counter()
    stage1_t0 = time.perf_counter()
    mem_before_stage1 = gpu_mem_str()
    smi_before_stage1 = nvidia_smi_snapshot()
    print(f"\n[Stage 1] Loading text encoder from {'GGUF(NF4) '+Path(cfg.gemma_gguf_path).name if cfg.gemma_gguf_path else 'bf16 safetensors'} ...")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    prompts_dir_p = Path(cfg.prompts_dir)
    prompt_files = sorted(prompts_dir_p.glob(cfg.prompts_glob))
    assert prompt_files, f"No prompt files matched {prompts_dir_p}/{cfg.prompts_glob}"
    engine = InferenceEngine(cfg)
    cached_per_file = engine.encode_all_prompts(prompt_files)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    stage1_s = time.perf_counter() - stage1_t0
    mem_after_stage1 = gpu_mem_str()
    smi_after_stage1 = nvidia_smi_snapshot()
    peak_stage1_gb = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f"[Stage 1] DONE in {stage1_s:.1f}s  peak={peak_stage1_gb:.2f}GB  mem {mem_before_stage1} -> {mem_after_stage1}")
    print(f"         smi {smi_before_stage1} -> {smi_after_stage1}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    stage2_t0 = time.perf_counter()
    mem_before_stage2 = gpu_mem_str()
    print(f"\n[Stage 2] Loading generator + VAEs from {engine._checkpoint} ...")
    gen_load_t0 = time.perf_counter()
    engine.load_generator()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    gen_load_s = time.perf_counter() - gen_load_t0
    mem_after_stage2 = gpu_mem_str()
    smi_after_stage2 = nvidia_smi_snapshot()
    peak_stage2_gb = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    dtype_counts, per_layer = dtype_histogram(engine.generator)
    print(f"[Stage 2] dtype histogram: {dtype_counts}")
    print(f"[Stage 2] top layers by params:")
    for name,dtype,n in per_layer[:8]:
        print(f"    {name:50s} {dtype:16s} {n/1e6:.1f}M")
    fp8_present = any("float8" in k for k in dtype_counts)
    print(f"[Stage 2] FP8 {'present' if fp8_present else 'NOT present (bf16 only — fp8_cast matched nothing!)'}")
    offload_info={}
    if cfg.sequential_offload_enabled:
        try:
            from ltx_core.model.transformer.sequential_offload import _find_transformer_blocks
            blocks = _find_transformer_blocks(engine.generator)
            offload_info={"n_blocks": len(blocks), "resident_gb": round(torch.cuda.memory_allocated()/1024**3,2), "first_block_device": str(next(blocks[0].parameters()).device)}
            print(f"[Stage 2] Offload: {offload_info}")
        except Exception as e: offload_info={"err":str(e)}
    stage2_s = time.perf_counter() - stage2_t0
    print(f"[Stage 2] DONE total {stage2_s:.1f}s (gen {gen_load_s:.1f}s)  peak={peak_stage2_gb:.2f}GB  mem {mem_before_stage2}->{mem_after_stage2}")
    print(f"         tiling={tiling_info['enabled']} tiles~{tiling_info.get('estimated_total_tiles','?')}")
    per_step_records = []
    original_base_generate = engine.base_pipeline.generate
    original_mem_generate = engine.memory_pipeline.generate
    gen_forward_times = []
    orig_gen_forward = engine.generator.forward
    def instrumented_forward(*args, **kwargs):
        t0=time.perf_counter()
        out = orig_gen_forward(*args, **kwargs)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        gen_forward_times.append(time.perf_counter()-t0)
        return out
    engine.generator.forward = instrumented_forward
    swap_records = []
    orig_log_stage = engine._log_stage
    def patched_log_stage(label, fn):
        t0=time.perf_counter()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        result = orig_log_stage(label, fn)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        swap_records.append({"label":label, "elapsed_s": round(elapsed,4), **gpu_mem_str()})
        return result
    engine._log_stage = patched_log_stage
    import ltx_distillation.utils as _utils
    orig_decode = _utils.decode_benchmark_sample
    decode_tile_records=[]
    def patched_decode(video_vae, audio_vae, video_latent, audio_latent, tiling_config=None):
        t0=time.perf_counter()
        out = orig_decode(video_vae, audio_vae, video_latent, audio_latent, tiling_config=tiling_config)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed=time.perf_counter()-t0
        decode_tile_records.append({"elapsed_s": round(elapsed,3), "tiling_enabled": tiling_config is not None, "tiling_config": str(tiling_config) if tiling_config else None, "video_latent_shape": list(video_latent.shape) if hasattr(video_latent,"shape") else None, **gpu_mem_str()})
        print(f"[Decode] patched_decode elapsed={elapsed:.2f}s tiling={tiling_config is not None}")
        return out
    _utils.decode_benchmark_sample = patched_decode
    import inference as _inf_mod
    _inf_mod.decode_benchmark_sample = patched_decode
    output_root = Path(cfg.output_root) / "outputs"
    shot_summaries=[]
    run_t0=time.perf_counter()
    for prompts_file in prompt_files:
        cached = cached_per_file.get(prompts_file, [])
        if not cached: continue
        prompt_name=prompts_file.stem
        timestamp=time.strftime("%Y%m%d_%H%M%S")
        run_output_dir=output_root / prompt_name / f"inference_{timestamp}"
        print(f"\n[Run] -> {run_output_dir}")
        gen_forward_times.clear(); swap_records.clear(); decode_tile_records.clear()
        profiler = None
        if do_profiler:
            from torch.profiler import profile, ProfilerActivity
            profiler = profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=False, with_stack=False)
            profiler.__enter__()
            print("[Profiler] started")
        shot_t0=time.perf_counter()
        try:
            engine.run_prompt_file(prompts_file, run_output_dir, cached)
        except Exception as e:
            print(f"run_prompt_file failed: {e}"); traceback.print_exc(); raise
        finally:
            if profiler is not None:
                profiler.__exit__(None,None,None)
                print("[Profiler] stopped")
                try:
                    profiler.export_chrome_trace(str(profiler_trace))
                    print(f"[Profiler] chrome trace -> {profiler_trace}")
                    events=profiler.key_averages()
                    buckets={"transfer":0,"attention":0,"mlp_linear":0,"other":0}
                    for e in events:
                        cuda_us=getattr(e,"device_time_total",0) or getattr(e,"cuda_time_total",0) or 0
                        name=e.key.lower()
                        t_us=max(cuda_us,0)
                        if any(k in name for k in ("memcpy","copy_","to_copy","pin_memory","htod","dtoh")): b="transfer"
                        elif any(k in name for k in ("attention","sdpa","flash","bmm","softmax","scaled_dot")): b="attention"
                        elif any(k in name for k in ("gemm","linear","addmm","matmul","mm_","cutlass","cublas")): b="mlp_linear"
                        else: b="other"
                        buckets[b]+=t_us/1e6
                    print(f"[Profiler] buckets (CUDA s): {buckets}")
                except Exception as e: print(f"[Profiler] export failed: {e}")
        shot_s=round(time.perf_counter()-shot_t0,2)
        shot_summaries.append({"prompts_file": str(prompts_file), "output_dir": str(run_output_dir), "shot_time_s": shot_s, "gen_forward_times_s": list(gen_forward_times), "avg_gen_forward_s": round(sum(gen_forward_times)/max(len(gen_forward_times),1),3) if gen_forward_times else None, "swap_records": list(swap_records), "decode_records": list(decode_tile_records)})
        print(f"[Run] shot batch done in {shot_s}s  avg gen forward {shot_summaries[-1]['avg_gen_forward_s']}s")
    overall_s=round(time.perf_counter()-overall_t0,2)
    if monitor: monitor.stop(); print(f"[GPU] polling stopped, {len(monitor.samples)} samples -> {gpu_csv}")
    with open(per_step_csv,"w",newline="") as f:
        w=csv.writer(f); w.writerow(["shot_idx","step_idx","gen_forward_s"])
        for si, summ in enumerate(shot_summaries):
            for step_idx, t in enumerate(summ["gen_forward_times_s"]):
                w.writerow([si, step_idx, t])
    print(f"[CSV] per-step -> {per_step_csv}")
    report={"run_id": run_id, "name": name, "seed": seed, "video": {"num_frames": num_frames, "height": video_height, "width": video_width, "fps": cfg.video_fps, "tokens": tokens, "video_shape": None, "audio_shape": None}, "system": sys_info, "config": {"fp8": use_fp8, "offload": use_offload, "tiling": tiling_info, "denoising_steps": list(cfg.denoising_steps), "denoising_sigmas": list(cfg.denoising_sigmas)}, "timing": {"overall_s": overall_s, "stage1_s": round(stage1_s,2), "stage2_s": round(stage2_s,2), "gen_load_s": round(gen_load_s,2), "peak_stage1_gb": round(peak_stage1_gb,2), "peak_stage2_gb": round(peak_stage2_gb,2)}, "gpu": {"before_stage1": mem_before_stage1, "after_stage1": mem_after_stage1, "after_stage2": mem_after_stage2, " dtype_histogram": dtype_counts, "per_layer_top": per_layer, "offload": offload_info}, "shots": shot_summaries, "artifacts": {"gpu_timeline": str(gpu_csv), "per_step": str(per_step_csv), "profiler_trace": str(profiler_trace) if do_profiler else None, "run_dir": str(run_profiling_dir)}}
    try:
        from ltx_distillation.utils import compute_latent_shapes
        vs, ash = compute_latent_shapes(num_frames=num_frames, video_height=video_height, video_width=video_width, batch_size=1, video_fps=float(cfg.video_fps))
        report["video"]["video_shape"]=list(vs); report["video"]["audio_shape"]=list(ash)
    except: pass
    report_json.write_text(json.dumps(report, indent=2))
    print(f"\n{'='*72}\nPROFILING REPORT -> {report_json}\n{json.dumps(report, indent=2)[:3000]}\n{'='*72}")
    outputs=sorted(glob.glob(f"{OUTPUT_DIR}/outputs/**/*.mp4", recursive=True), key=lambda p: Path(p).stat().st_mtime)
    video_path = outputs[-1] if outputs else None
    if video_path:
        print(f"Latest output: {video_path}")
        if preview:
            try: display(Video(video_path, width=640))
            except Exception as e: print(f"Preview failed: {e}")
        if os.path.isdir("/content/drive/MyDrive"):
            keep_dir="/content/drive/MyDrive/joyai-echo-outputs"
            os.makedirs(keep_dir, exist_ok=True)
            kept=shutil.copy(video_path, keep_dir)
            print(f"Copied to Drive: {kept}")
    else: print("No .mp4 produced — check logs")
    return video_path, report_json, run_profiling_dir
print("generate_video_profiling() ready — call it in the next cell")
print("   Example: video_path, report, prof_dir = generate_video_profiling(prompts, name='french_lesson', num_frames=497, video_height=736, video_width=1280)")


In [ ]:
prompts = [
    "PIERRE the sky-blue cartoon parrot lands on DINO the mint-green baby dinosaur's head. In a cheerful voice, Pierre says, \"Bonjour, Dino!\" Bright 2D cartoon style, sunny meadow background, cheerful children's music.",
]
SEED_VIDEO_PATH = None
USE_SMALL = False
if USE_SMALL:
    video_path, report_json, prof_dir = generate_video_profiling(
        prompts, name="profiling_small", seed=42,
        num_frames=121, video_height=480, video_width=832,
        seed_video=SEED_VIDEO_PATH,
        enable_profiler=False, enable_gpu_polling=True,
    )
else:
    video_path, report_json, prof_dir = generate_video_profiling(
        prompts, name="french_lesson_profiling", seed=42,
        num_frames=497, video_height=736, video_width=1280,
        seed_video=SEED_VIDEO_PATH,
        enable_profiler=ENABLE_TORCH_PROFILER, enable_gpu_polling=ENABLE_GPU_POLLING,
    )
print(f"\nRun done\n  video: {video_path}\n  report: {report_json}\n  dir: {prof_dir}")


In [ ]:
import json, glob, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
reports = sorted(Path(PROFILING_DIR).rglob("profiling_report.json"), key=lambda p: p.stat().st_mtime)
if not reports:
    print("No profiling_report.json found — run Cell 7 first")
else:
    report_path = reports[-1]
    report = json.loads(report_path.read_text())
    print(f"Latest report: {report_path}")
    print(json.dumps({k: report[k] for k in ("run_id","timing","config")}, indent=2))
    fig, axes = plt.subplots(1,3, figsize=(16,4))
    axes[0].bar(["Stage1\n(text enc)","Stage2\n(gen+VAE)","Shots\n(denoise+decode)"], [report["timing"]["stage1_s"], report["timing"]["stage2_s"], sum(s["shot_time_s"] for s in report["shots"])], color=["#4e79a7","#f28e2b","#59a14f"])
    axes[0].set_title("Wall time by stage (s)"); axes[0].set_ylabel("seconds")
    for i,v in enumerate([report["timing"]["stage1_s"], report["timing"]["stage2_s"], sum(s["shot_time_s"] for s in report["shots"])]):
        axes[0].text(i, v+1, f"{v:.1f}s", ha="center", fontsize=9)
    all_steps = []
    for si,s in enumerate(report["shots"]):
        all_steps.extend(s.get("gen_forward_times_s",[]))
    if all_steps:
        axes[1].bar(range(len(all_steps)), all_steps, color="#e15759")
        axes[1].set_title(f"Per denoising forward (s) — {len(all_steps)} steps, avg {sum(all_steps)/len(all_steps):.2f}s")
        axes[1].set_xlabel("step idx (flattened across shots)"); axes[1].set_ylabel("seconds")
    else:
        axes[1].text(0.5,0.5,"No gen_forward_times\n(check Cell 6 ran?)", ha="center", transform=axes[1].transAxes)
    swaps=[]
    labels=[]
    for s in report["shots"]:
        for r in s.get("swap_records",[]):
            swaps.append(r["elapsed_s"]*1000); labels.append(r["label"][:18])
    if swaps:
        axes[2].barh(range(len(swaps)), swaps, color="#76b7b2")
        axes[2].set_yticks(range(len(swaps))); axes[2].set_yticklabels(labels, fontsize=7)
        axes[2].set_title("Stage-swap latency (ms)"); axes[2].set_xlabel("ms")
    else:
        axes[2].text(0.5,0.5,"No swap records", ha="center", transform=axes[2].transAxes)
    plt.tight_layout(); plt.show()
    gpu_csv = Path(report["artifacts"]["gpu_timeline"])
    if gpu_csv.exists():
        df = pd.read_csv(gpu_csv)
        print(f"\nGPU timeline: {len(df)} samples, {df['elapsed_s'].max():.0f}s span")
        print(df.describe().round(1).to_string())
        fig, ax1 = plt.subplots(figsize=(14,4))
        ax1.plot(df["elapsed_s"], df["alloc_gb"], label="alloc GB", color="#4e79a7")
        ax1.plot(df["elapsed_s"], df["reserved_gb"], label="reserved GB", color="#59a14f", linestyle="--")
        ax1.plot(df["elapsed_s"], df["mem_used_gb"], label="nvidia-smi used GB", color="#f28e2b", alpha=0.7)
        ax1.set_xlabel("elapsed (s)"); ax1.set_ylabel("GB"); ax1.legend(loc="upper left")
        ax2 = ax1.twinx()
        ax2.plot(df["elapsed_s"], df["gpu_util"], label="GPU util %", color="#e15759", alpha=0.5)
        ax2.set_ylabel("GPU util %", color="#e15759")
        plt.title("GPU memory + utilization over run"); plt.tight_layout(); plt.show()
        print("\nTiling:", report["config"]["tiling"])
        print("\nDtypes:", report["gpu"][" dtype_histogram"])
        print("   Offload:", report["gpu"]["offload"])
    else:
        print(f"GPU timeline CSV not found: {gpu_csv}")
    total = report["timing"]["overall_s"]
    s1 = report["timing"]["stage1_s"]; s2 = report["timing"]["stage2_s"]
    shots_s = sum(s["shot_time_s"] for s in report["shots"])
    print(f"\n{'='*60}\nBOTTLENECK SUMMARY\n  Overall: {total:.1f}s  Stage1 {s1/total*100:.1f}%  Stage2 {s2/total*100:.1f}%  Shots {shots_s/total*100:.1f}%")
    if all_steps:
        print(f"  Avg gen forward: {sum(all_steps)/len(all_steps):.2f}s  ({len(all_steps)} forwards, {sum(all_steps):.1f}s total)")
        print(f"  -> Denoise is ~{sum(all_steps)/max(shots_s,1)*100:.0f}% of shot time (rest is decode + swaps)")
    summary_md = report_path.parent / "bottleneck_summary.md"
    summary_md.write_text(f"# Bottleneck {report['run_id']}\n\n- Overall {total:.1f}s | Stage1 {s1:.1f}s | Stage2 {s2:.1f}s | Shots {shots_s:.1f}s\n- Tokens {report['video']['tokens'] if 'tokens' in report['video'] else '?'} | Tiling {report['config']['tiling']}\n- Dtypes {report['gpu'][' dtype_histogram']}\n- Offload {report['gpu']['offload']}\n")
    print(f"Summary -> {summary_md}")
    print("\nAll artifacts in", report_path.parent)
    for p in sorted(report_path.parent.iterdir()):
        print(f"  {p.name:30s} {p.stat().st_size/1024:.0f} KB")


In [ ]:
RUN_SINGLE_STEP_PROFILER = False
if not RUN_SINGLE_STEP_PROFILER:
    print("Skipped — set RUN_SINGLE_STEP_PROFILER=True to run")
else:
    import os, sys, time, json
    from pathlib import Path
    from torch.profiler import ProfilerActivity, profile
    FRAMES = 121
    HEIGHT, WIDTH = 480, 832
    OUT_JSON = str(Path(PROFILING_DIR)) + f"/single_step_profile_{FRAMES}_{HEIGHT}x{WIDTH}.json"
    OFFLOAD = True
    from inference import InferenceConfig, InferenceEngine
    gguf = Path("/content/gguf_path.txt").read_text().strip()
    print(f"[profile] frames={FRAMES} {HEIGHT}x{WIDTH} offload={OFFLOAD} gguf={Path(gguf).name}")
    cfg = InferenceConfig(f"{REPO_ROOT}/configs/inference.yaml", num_frames=FRAMES, video_height=HEIGHT, video_width=WIDTH, gemma_gguf_path=gguf, quantization_fp8_enabled=True, sequential_offload_enabled=OFFLOAD, prompts_glob="test_001.json")
    engine = InferenceEngine(cfg)
    t0=time.perf_counter()
    cached = engine.encode_all_prompts([Path(f"{REPO_ROOT}/prompts/test_001.json")])
    print(f"[profile] stage1 {time.perf_counter()-t0:.1f}s")
    t0=time.perf_counter()
    engine.load_generator()
    print(f"[profile] stage2 {time.perf_counter()-t0:.1f}s")
    tokens = ((FRAMES-1)//8+1)*(HEIGHT//32)*(WIDTH//32)
    print(f"[profile] tokens={tokens}  peak={torch.cuda.max_memory_allocated()/1024**3:.2f}GB")
    engine._stage_for_denoise()
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    pipe = engine.base_pipeline
    cond = {k: (v.to(engine.device) if isinstance(v, torch.Tensor) else v) for k,v in list(cached.values())[0][0].items()}
    from ltx_distillation.utils import compute_latent_shapes
    video_shape, audio_shape = compute_latent_shapes(num_frames=FRAMES, video_height=HEIGHT, video_width=WIDTH, batch_size=1, video_fps=float(cfg.video_fps))
    print(f"[profile] video_shape={video_shape} audio_shape={audio_shape}")
    device, dtype = engine.device, engine.dtype
    video = torch.randn(video_shape, device=device, dtype=dtype)
    audio = torch.randn(audio_shape, device=device, dtype=dtype)
    sigma = pipe.denoising_sigmas[0]
    B, F_v, F_a = video_shape[0], video_shape[1], audio_shape[1]
    v_sigma = sigma*torch.ones([B,F_v], device=device)
    a_sigma = sigma*torch.ones([B,F_a], device=device)
    print("[profile] warm-up...")
    with torch.no_grad():
        pipe.generator(noisy_image_or_video=video, conditional_dict=cond, timestep=v_sigma, noisy_audio=audio, audio_timestep=a_sigma)
    torch.cuda.synchronize()
    print("[profile] profiling one forward...")
    t0=time.perf_counter()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=False, profile_memory=False, with_stack=False) as prof:
        with torch.no_grad():
            pipe.generator(noisy_image_or_video=video, conditional_dict=cond, timestep=v_sigma, noisy_audio=audio, audio_timestep=a_sigma)
        torch.cuda.synchronize()
    step_s=time.perf_counter()-t0
    print(f"[profile] step {step_s:.2f}s  peak {torch.cuda.max_memory_allocated()/1024**3:.2f}GB")
    events=prof.key_averages()
    buckets={"transfer":0,"attention":0,"mlp_linear":0,"other":0}
    rows=[]
    for e in events:
        cuda_us=getattr(e,"device_time_total",0) or getattr(e,"cuda_time_total",0) or 0
        cpu_us=e.cpu_time_total or 0
        name=e.key.lower()
        t_us=max(cuda_us,0)
        if any(k in name for k in ("memcpy","copy_","to_copy","pin_memory","htod","dtoh")): b="transfer"
        elif any(k in name for k in ("attention","sdpa","flash","bmm","softmax","scaled_dot")): b="attention"
        elif any(k in name for k in ("gemm","linear","addmm","matmul","mm_","cutlass","cublas")): b="mlp_linear"
        else: b="other"
        buckets[b]+=t_us/1e6
        rows.append((e.key, cuda_us/1e6, cpu_us/1e6))
    total_b=sum(buckets.values())
    print("\n[profile] BUCKETS (CUDA s)")
    for k,v in sorted(buckets.items(), key=lambda x: -x[1]):
        print(f"  {k:12s} {v:7.2f}s  ({v/max(total_b,1e-9)*100:4.1f}%)")
    print(f"  {'SUM':12s} {total_b:7.2f}s  wall {step_s:.2f}s  util {total_b/max(step_s,1e-9)*100:.0f}%")
    print("\n[profile] TOP 25 kernels")
    for name,cu,cp in sorted(rows, key=lambda r: -r[1])[:25]:
        print(f"  {cu:7.3f}s cuda | {cp:7.3f}s cpu | {name[:85]}")
    json.dump({"frames":FRAMES,"height":HEIGHT,"width":WIDTH,"tokens":tokens,"step_s":step_s,"buckets":buckets, "peak_gb":torch.cuda.max_memory_allocated()/1024**3, "top_kernels":[{"name":n,"cuda_s":c,"cpu_s":p} for n,c,p in sorted(rows, key=lambda r: -r[1])[:40]]}, open(OUT_JSON,"w"), indent=2)
    print(f"\n[profile] wrote {OUT_JSON}")


In [ ]:
import shutil, glob
from pathlib import Path
reports = sorted(Path(PROFILING_DIR).rglob("profiling_report.json"), key=lambda p: p.stat().st_mtime)
if not reports:
    print("No reports yet")
else:
    latest_dir = reports[-1].parent
    print(f"Latest profiling dir: {latest_dir}")
    for p in sorted(latest_dir.iterdir()):
        print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")
    zip_path = f"/content/{latest_dir.name}.zip"
    shutil.make_archive(f"/content/{latest_dir.name}", "zip", latest_dir)
    print(f"\nZipped -> {zip_path} ({Path(zip_path).stat().st_size/1024/1024:.1f} MB)")
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e: print(f"Download failed: {e} — file is at {zip_path}, also copied to Drive below")
    if Path("/content/drive/MyDrive").exists():
        drive_out = Path("/content/drive/MyDrive/joyai-echo-outputs") / latest_dir.name
        shutil.copytree(latest_dir, drive_out, dirs_exist_ok=True)
        shutil.copy(zip_path, drive_out.parent / Path(zip_path).name)
        print(f"Copied to Drive: {drive_out}")
